In [6]:
import pandas as pd
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 150)

# load the raw data
df = pd.read_csv("data/raw/support2.csv")
print("Loaded:", df.shape)

# check for dupes before we do anything to the data
raw_dupes = df.duplicated().sum()
print("Duplicate rows in raw data:", raw_dupes)

df = df.rename(columns={'d.time': 'd_time', 'num.co': 'num_co'})
TARGET = 'hospdead'

leakage_cols = ['death', 'd_time', 'slos', 'surv2m', 'surv6m', 'prg2m', 'prg6m', 'sfdm2', 'hday']
drop_cols = [c for c in leakage_cols if c in df.columns]
df_model = df.drop(columns=drop_cols)

# tracking shape at each step
shape_log = {}
shape_log['raw'] = df.shape
shape_log['after_leakage_drop'] = df_model.shape

print(f"Dropped {len(drop_cols)} leakage/administrative columns: {drop_cols}")
print(f"Remaining shape: {df_model.shape}")



Loaded: (9105, 47)
Duplicate rows in raw data: 0
Dropped 9 leakage/administrative columns: ['death', 'd_time', 'slos', 'surv2m', 'surv6m', 'prg2m', 'prg6m', 'sfdm2', 'hday']
Remaining shape: (9105, 38)


In [7]:
# quick sanity check on the categorical columns (look for typos, weird captials etc). 
cat_check_cols = df_model.select_dtypes(include='object').columns.tolist()

print("Unique values per categorical column (excluding NaN):\n")
for c in cat_check_cols:
    vals = df_model[c].dropna().unique()
    print(f"{c} ({len(vals)} unique): {sorted(vals.tolist())[:10]}{' ...' if len(vals) > 10 else ''}")

print("\nAge range: min =", df_model['age'].min(), " max =", df_model['age'].max())
# looks fine, nothing weird here

# these can't actually be 0 or negative in real life & turn them into NaN so they get handled properly
for c in ['meanbp', 'hrt', 'resp']:
    df_model.loc[df_model[c] == 0, c] = pd.NA

for c in ['totmcst', 'dnrday']:
    df_model.loc[df_model[c] < 0, c] = pd.NA



Unique values per categorical column (excluding NaN):

sex (2 unique): ['female', 'male']
dzgroup (8 unique): ['ARF/MOSF w/Sepsis', 'CHF', 'COPD', 'Cirrhosis', 'Colon Cancer', 'Coma', 'Lung Cancer', 'MOSF w/Malig']
dzclass (4 unique): ['ARF/MOSF', 'COPD/CHF/Cirrhosis', 'Cancer', 'Coma']
income (4 unique): ['$11-$25k', '$25-$50k', '>$50k', 'under $11k']
race (5 unique): ['asian', 'black', 'hispanic', 'other', 'white']
ca (3 unique): ['metastatic', 'no', 'yes']
dnr (3 unique): ['dnr after sadm', 'dnr before sadm', 'no dnr']

Age range: min = 18.04199  max = 101.84796


C:\Users\srika\AppData\Local\Temp\ipykernel_26744\3957361379.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_check_cols = df_model.select_dtypes(include='object').columns.tolist()


In [8]:
# Deal with missing values 
missing_pct = df_model.isnull().mean().sort_values(ascending=False) * 100
print(missing_pct[missing_pct > 40])

num_cols = [c for c in df_model.select_dtypes(include='number').columns if c != TARGET]
cat_cols = df_model.select_dtypes(include='str').columns.tolist()
if not cat_cols:
    cat_cols = df_model.select_dtypes(include='object').columns.tolist()

# For columns missing a ton of data, add a flag column before filling it in
# (the fact it was missing might matter, don't want to lose that info)
for c in num_cols:
    if df_model[c].isnull().any():
        if missing_pct.get(c, 0) > 40:
            df_model[f'{c}_was_missing'] = df_model[c].isnull().astype(int)
        df_model[c] = df_model[c].fillna(df_model[c].median())

# Fill with "Missing" 
for c in cat_cols:
    if df_model[c].isnull().any():
        df_model[c] = df_model[c].fillna('Missing')

# Double check data
assert df_model.isnull().sum().sum() == 0
shape_log['after_missing_handling'] = df_model.shape
print("All missing values resolved. Shape:", df_model.shape)


adlp       61.954970
urine      53.399231
glucose    49.423394
bun        47.797913
dtype: float64
All missing values resolved. Shape: (9105, 42)


In [9]:
# Check for extreme values
for c in ['age', 'charges', 'totcst', 'num_co']:
    q1, q99 = df_model[c].quantile([0.01, 0.99])
    print(f"{c}: 1st pct={q1:.1f}, 99th pct={q99:.1f}, max={df_model[c].max():.1f}")

# Check dupes again since dropping columns could've created some
post_clean_dupes = df_model.duplicated().sum()
print("\nDuplicate rows after leakage removal + missing-value handling:", post_clean_dupes)


age: 1st pct=22.6, 99th pct=91.8, max=101.8
charges: 1st pct=2389.0, 99th pct=508771.4, max=1435423.0
totcst: 1st pct=1645.7, 99th pct=215790.1, max=633212.0
num_co: 1st pct=0.0, 99th pct=6.0, max=9.0

Duplicate rows after leakage removal + missing-value handling: 0


In [10]:
# Encode sex as 0/1
df_model['sex'] = df_model['sex'].map({'male': 0, 'female': 1})

# Encode income (has an actual order)
income_order = ['under $11k', '$11-$25k', '$25-$50k', '>$50k']
income_map = {label: i for i, label in enumerate(income_order)}

df_model['income_was_missing'] = (df_model['income'] == 'Missing').astype(int)
df_model['income'] = df_model['income'].map(income_map)
median_income_code = df_model['income'].median()
df_model['income'] = df_model['income'].fillna(median_income_code).astype(int)

# No natural order
nominal_cols = [c for c in ['dzgroup', 'dzclass', 'race', 'ca', 'dnr'] if c in df_model.columns]
df_model = pd.get_dummies(df_model, columns=nominal_cols, drop_first=True)

shape_log['after_encoding'] = df_model.shape
print("Final cleaned/encoded shape:", df_model.shape)
print(df_model.dtypes.value_counts())

# Recap of shape changes at each step
summary_df = pd.DataFrame(shape_log, index=['rows', 'columns']).T
print(summary_df)

# Save it
df_model.to_csv('data/processed/support2_cleaned.csv', index=False)
print("Saved -> data/processed/support2_cleaned.csv")



Final cleaned/encoded shape: (9105, 58)
float64    27
bool       20
int64      11
Name: count, dtype: int64
                        rows  columns
raw                     9105       47
after_leakage_drop      9105       38
after_missing_handling  9105       42
after_encoding          9105       58
Saved -> data/processed/support2_cleaned.csv
